# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [37]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [38]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

MODEL_OPTIONS = {
    "OpenAI (gpt-4o-mini)": {"provider": "openai", "model": MODEL_GPT},
    "Ollama (llama3.2)": {"provider": "ollama", "model": MODEL_LLAMA},
}

In [3]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI(
    base_url=base_url,
    api_key=api_key
)

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


In [39]:
# Default system prompt and starter question

system_role = """
You are a senior software engineer and technical mentor.
Explain technical concepts clearly, with practical examples and trade-offs.
Use concise language, but include enough detail for someone learning.
If code is provided, explain what it does, why it works, and potential pitfalls.
"""

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [50]:
def stream_expert_answer(message, history, model_choice, expert_system_prompt):
    selected = MODEL_OPTIONS[model_choice]
    client = openai if selected["provider"] == "openai" else ollama

    messages = [{"role": "system", "content": (expert_system_prompt or system_role).strip()}]
    for item in history or []:
        role = item.get("role")
        if role in ("user", "assistant"):
            messages.append({"role": role, "content": item.get("content", "")})
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=selected["model"],
        messages=messages,
        stream=True,
    )

    partial = ""
    for chunk in stream:
        if not getattr(chunk, "choices", None):
            continue
        delta = chunk.choices[0].delta
        text = getattr(delta, "content", None) or ""
        if text:
            partial += text
            yield partial

In [51]:
# Build a simple Gradio UI with model switch + editable system prompt
model_dropdown = gr.Dropdown(
    choices=list(MODEL_OPTIONS.keys()),
    value="OpenAI (gpt-4o-mini)",
    label="Choose model",
)

system_prompt_box = gr.Textbox(
    value=system_role.strip(),
    label="System Prompt (expertise)",
    lines=5,
)

demo = gr.ChatInterface(
    fn=stream_expert_answer,
    type="messages",
    additional_inputs=[model_dropdown, system_prompt_box],
    title="Technical Question Answerer",
    description="Ask technical questions, stream responses, switch models, and tune expertise via the system prompt.",
)
# Launch the app
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
